# Data operations

In [5]:
command = f"kubectl get svc | awk '$1 == \"postgresql\" {{print $3}}'"

# Execute the command
result = subprocess.run(command, shell=True, capture_output=True, text=True)

# Get the output
ip = result.stdout.strip()

In [6]:
ip

'10.96.1.190'

In [7]:
port=5432

In [8]:
user="postgres"
database="postgres"
password="postgres"

In [9]:
print(ip)
print(port)
print(password)

10.96.1.190
5432
postgres


In [10]:
!pip show tensorflow

Name: tensorflow
Version: 2.17.1
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: /opt/conda/lib/python3.11/site-packages
Requires: absl-py, astunparse, flatbuffers, gast, google-pasta, grpcio, h5py, keras, libclang, ml-dtypes, numpy, opt-einsum, packaging, protobuf, requests, setuptools, six, tensorboard, tensorflow-io-gcs-filesystem, termcolor, typing-extensions, wrapt
Required-by: 


In [11]:
import os
# Suppress TensorFlow logging and oneDNN info
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
# Force legacy Keras behavior to prevent the AttributeError
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import numpy as np
import pandas as pd
import psycopg2
from sklearn.model_selection import train_test_split

# Verify versions without crashing
print(f"TensorFlow: {tf.__version__}")

TensorFlow: 2.17.1


In [12]:
# data=keras.datasets.california_housing.load_data(
#     version="large", path="california_housing.npz", test_split=0.2, seed=113
# )

### Copy california_housing.npz to keras home

In [13]:
!pwd

/home/rajesh-babu-muda/PCAI-D-Pipelines/PCAI-Pipelines-main


In [14]:
!cp /home/rajesh-babu-muda/PCAI-D-Pipelines/PCAI-Pipelines-main/01-Data-Operations/california_housing.npz /home/rajesh-babu-muda/.keras/datasets/

In [15]:
#!cp /home/dhanunjaya-rao.nethala-hpe.com/PCAI-Pipelines-main/01-Data-Operations/california_housing.npz ~/.keras/datasets/

In [16]:
def load_data_offline(file_path):
    """Loads California Housing data from a local .npz file."""
    # Load the numpy archive
    with np.load(file_path) as data:
        # The file contains: ['data_train', 'target_train', 'data_test', 'target_test']
        x = data["data_train"]
        y = data["target_train"]
    
    # Split the data locally
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=0.2, random_state=113
    )
    
    column_names = [
        'Longitude', 'Latitude', 'Median_Housing_Age', 'Total_Rooms', 
        'Total_Bedrooms', 'Population', 'Households', 'Median_Income'
    ]
    
    # Create DataFrames
    df_train = pd.DataFrame(x_train, columns=column_names)
    df_train['Price'] = y_train
    
    df_test = pd.DataFrame(x_test, columns=column_names)
    df_test['Price'] = y_test
    
    return df_train, df_test

# Use the path defined in your notebook
local_path = '/home/rajesh-babu-muda/.keras/datasets/california_housing.npz'
df_train, df_test = load_data_offline(local_path)

print(df_train.head())

    Longitude   Latitude  Median_Housing_Age  Total_Rooms  Total_Bedrooms  \
0 -117.190002  34.049999                33.0       3007.0           498.0   
1 -118.239998  33.910000                37.0       1607.0           377.0   
2 -117.150002  32.919998                16.0       2969.0           514.0   
3 -122.220001  37.810001                52.0       2944.0           536.0   
4 -121.809998  37.250000                20.0       3398.0           771.0   

   Population  Households  Median_Income     Price  
0      1252.0       488.0         3.8816  134600.0  
1      1526.0       375.0         1.7158   94300.0  
2      1594.0       465.0         4.5221  168300.0  
3      1034.0       521.0         5.3509  302100.0  
4      1231.0       744.0         2.0288  350000.0  


In [18]:
total_rooms = df_train.iloc[:, 3]  # Total_Rooms
total_bedrooms = df_train.iloc[:, 4]  # Total_Bedrooms
population = df_train.iloc[:, 5]  # Population
households = df_train.iloc[:, 6]  # Households

#New columns
room_per_house = total_rooms / households
bedrooms_ratio = total_bedrooms / total_rooms
people_per_house = population / households
df_train['Room per House'] = room_per_house
df_train['Bedrooms Ratio'] = bedrooms_ratio
df_train['People per House'] = people_per_house
print(df_train.head())

    Longitude   Latitude  Median_Housing_Age  Total_Rooms  Total_Bedrooms  \
0 -117.190002  34.049999                33.0       3007.0           498.0   
1 -118.239998  33.910000                37.0       1607.0           377.0   
2 -117.150002  32.919998                16.0       2969.0           514.0   
3 -122.220001  37.810001                52.0       2944.0           536.0   
4 -121.809998  37.250000                20.0       3398.0           771.0   

   Population  Households  Median_Income     Price  Room per House  \
0      1252.0       488.0         3.8816  134600.0        6.161885   
1      1526.0       375.0         1.7158   94300.0        4.285333   
2      1594.0       465.0         4.5221  168300.0        6.384946   
3      1034.0       521.0         5.3509  302100.0        5.650672   
4      1231.0       744.0         2.0288  350000.0        4.567204   

   Bedrooms Ratio  People per House  
0        0.165614          2.565574  
1        0.234599          4.069334  
2 

In [51]:

# Conexion settings
conn = psycopg2.connect(
    host=ip,
    port=port,
    database=database,
    user=user,
    password=password
)
cur = conn.cursor()

create_table_query = '''
CREATE TABLE  california_housing (
    Longitude FLOAT,
    Latitude FLOAT,
    Median_Housing_Age FLOAT,
    Total_Rooms FLOAT,
    Total_Bedrooms FLOAT,
    Population FLOAT,
    Households FLOAT,
    Median_Income FLOAT,
    Price FLOAT, 
    RoomPerHouse FLOAT,
    BedroomRatio FLOAT,
    PeoplePerHouse FLOAT
    
);
'''
cur.execute(create_table_query)
conn.commit()




In [52]:
for _, row in df_train.iterrows():
    insert_query = '''
    INSERT INTO california_housing (Longitude, Latitude, Median_Housing_Age, Total_Rooms, 
    Total_Bedrooms, Population, Households, Median_Income, Price,RoomPerHouse,BedroomRatio,PeoplePerHouse)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    '''
    cur.execute(insert_query, tuple(row))

# commit the changes
conn.commit()

# close connection
cur.close()
conn.close()

In [53]:

conn = psycopg2.connect(
    host=ip,
    port=port,
    database=database,
    user=user,
    password=password
)


cur = conn.cursor()

# query
cur.execute("SELECT * FROM california_housing")


column_names = [desc[0] for desc in cur.description]
rows = cur.fetchall()


print("Columns:", column_names)

print("Values: ",rows[:1])


# close conexion
cur.close()
conn.close()

Columns: ['longitude', 'latitude', 'median_housing_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'price', 'roomperhouse', 'bedroomratio', 'peopleperhouse']
Values:  [(-117.19000244140625, 34.04999923706055, 33.0, 3007.0, 498.0, 1252.0, 488.0, 3.8815999031066895, 134600.0, 6.1618852615356445, 0.1656135618686676, 2.5655736923217773)]


In [22]:
print("your data conector: ")
print(f'URI: "jdbc:postgresql://{ip}:{port}/postgres"')
print(f'Username: "{user}"')
print(f'Password: "{password}"')

your data conector: 
URI: "jdbc:postgresql://10.103.7.141:5432/postgres"
Username: "postgres"
Password: "postgres"
